In [1]:
import sys, platform, importlib.metadata, json
print("AUDIT_KERNEL="+json.dumps({"python":sys.version,"executable":sys.executable,"platform":platform.platform(),"packages":dict(sorted((d.metadata["Name"],d.version) for d in importlib.metadata.distributions()))}))

AUDIT_KERNEL={"python": "3.12.10 (tags/v3.12.10:0cc8128, Apr  8 2025, 12:21:36) [MSC v.1943 64 bit (AMD64)]", "executable": "C:\\Users\\Miakhlov\\Documents\\GitHub\\Computational chemistry_course\\tmp\\course-restructure-20261118\\scripts\\environment\\.venv-chem\\Scripts\\python.exe", "platform": "Windows-10-10.0.19045-SP0", "packages": {"Jinja2": "3.1.6", "MarkupSafe": "3.0.4", "PyYAML": "6.0.3", "Pygments": "2.21.0", "Send2Trash": "2.2.0", "altair": "6.3.0", "anyio": "4.16.0", "argon2-cffi": "25.1.0", "argon2-cffi-bindings": "26.2.0", "arrow": "1.4.0", "asttokens": "3.0.2", "async-lru": "2.4.0", "attrs": "26.1.0", "babel": "2.18.0", "beautifulsoup4": "4.15.0", "bleach": "6.4.0", "certifi": "2026.10.23", "cffi": "2.1.1", "charset-normalizer": "3.5.2", "click": "8.5.1", "cloudpickle": "3.1.3", "colorama": "0.4.6", "comm": "0.2.3", "contourpy": "1.4.0", "cycler": "0.12.1", "debugpy": "1.8.22", "defusedxml": "0.7.1", "executing": "2.3.0", "fastjsonschema": "2.22.3", "fonttools": "4.68.0

# Разбор A. Структурное сходство

Ксантины имеют общий мотив, но список «похожих» зависит от определения сходства. Проверим, насколько рейтинг к кофеину устойчив к радиусу Morgan. Гипотеза: локальная похожесть не совпадает с равенством молекул. Для B кислота и лактон могут иметь высокий коэффициент, оставаясь разными аналитическими объектами.

Получите fingerprints для четырёх ксантинов при radius=2, chirality=True. Вычислите полную матрицу, проверьте симметрию и единицы на диагонали. Измените радиус на 3 и сравните рейтинг. Отдельный ручной пример двух множеств проверяет формулу вне RDKit.

In [2]:
from pathlib import Path
import sys,json,math
from IPython.display import display
ROOT=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'programm/course.json').is_file())
sys.path.insert(0,str(ROOT/'practicuum/shared'))
from course_support import DATA,compound,esol,workspace,write_json,init_dossier
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
def publish_plot(fig,name):
    fig.savefig(OUT/name,dpi=120,bbox_inches='tight')
    display(fig)
    plt.close(fig)
OUT=workspace('M05-L01','case')


Сначала проверяем входы и единицы. Словари сохраняют названия полей вместе со значениями; список сохраняет соответствие записей. Выведите промежуточный результат перед дальнейшим анализом.

In [3]:
from rdkit import Chem,DataStructs
from rdkit.Chem import rdFingerprintGenerator
import numpy as np
names=['caffeine','theophylline','theobromine','istradefylline']
def matrix(radius):
    g=rdFingerprintGenerator.GetMorganGenerator(radius=radius,fpSize=2048,includeChirality=True)
    fps=[g.GetFingerprint(Chem.MolFromSmiles(compound(n)['smiles'])) for n in names]
    return np.array([[DataStructs.TanimotoSimilarity(a,b) for b in fps] for a in fps])
values=matrix(2);assert np.allclose(values,values.T) and np.allclose(np.diag(values),1)
display(values)


array([[1.        , 0.45714286, 0.52941176, 0.22033898],
       [0.45714286, 1.        , 0.29268293, 0.19672131],
       [0.52941176, 0.29268293, 1.        , 0.17460317],
       [0.22033898, 0.19672131, 0.17460317, 1.        ]])

Теперь сравним модель с контрольным случаем. `assert` останавливает расчёт при нарушении контракта; это защита от ошибочного продолжения, а не оценка научного понимания.

In [4]:
fig,ax=plt.subplots();im=ax.imshow(values,vmin=0,vmax=1);fig.colorbar(im,ax=ax,label='Tanimoto')
ax.set_xticks(range(4),names,rotation=25);ax.set_yticks(range(4),names);publish_plot(fig,'similarity.png')
print('radius=3 caffeine similarities',matrix(3)[0].tolist())


<Figure size 640x480 with 2 Axes>

radius=3 caffeine similarities [1.0, 0.3333333333333333, 0.4, 0.15294117647058825]


Сохраним именно полученный результат. Не переносите этот вывод на другую химическую форму, мишень или экспериментальные условия без отдельной проверки.

In [5]:
a={1,2,3};b={2,3,4,5};assert len(a&b)/len(a|b)==.4
result={'names':names,'matrix':values.tolist(),'kind':'calculated','radius':2,'fpSize':2048,'includeChirality':True}


In [6]:
write_json(OUT/'result.json',result)
from dossier_records import append_record
receipt=append_record(init_dossier(),'M05-L01','A',result,ROOT,inputs=['practicuum/shared/data/compounds.json'])
print('Versioned evidence:',receipt)


Versioned evidence: C:\Users\Miakhlov\Documents\GitHub\Computational chemistry_course\tmp\course-restructure-20261118\practicuum\evidence\20261118-full-course-windows\workspace\my_research\evidence\M05-L01\20261118T141549780960Z-A-2588ea845fde49acb81cc6376a5883cb.json


Изменение рейтинга является свойством представления. Чтобы выбрать следующий опыт, нужно учитывать не только сходство, но и неизвестность. Следующий урок добавит несколько критериев отбора.

Перед переходом ответьте: какие результаты получены вычислением, какие заданы условно, и какая независимая проверка ещё отсутствует?